In [2]:
# Import libraries
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

# Enable experimental IterativeImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print("✅ Libraries imported!")

✅ Libraries imported!


In [3]:
# Load the raw data (same as before)
def load_csv_robust(filepath, skip_rows=20):
    """Load CSV with manual parsing to handle copyright header"""
    import csv
    data = []
    header = None
    
    with open(filepath, 'r', encoding='utf-8') as f:
        # Skip the header lines
        for _ in range(skip_rows):
            f.readline()
        
        # Read the header
        header_line = f.readline().strip()
        header = header_line.split(',')
        
        # Read the data
        reader = csv.reader(f)
        for row in reader:
            if not row or len(row) < 2:
                continue
            if len(row) < len(header):
                row.extend([''] * (len(header) - len(row)))
            elif len(row) > len(header):
                row = row[:len(header)]
            data.append(row)
    
    df = pd.DataFrame(data, columns=header)
    
    for col in df.columns:
        if col != 'class':
            df[col] = df[col].replace('na', np.nan)
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

# Load data
print("📂 Loading data...")
train_df = load_csv_robust('../data/raw/train.csv', skip_rows=20)
test_df = load_csv_robust('../data/raw/test.csv', skip_rows=20)

# Separate features and target
X_train_raw = train_df.drop('class', axis=1)
y_train = train_df['class'].map({'neg': 0, 'pos': 1})

X_test_raw = test_df.drop('class', axis=1) if 'class' in test_df.columns else test_df
y_test = test_df['class'].map({'neg': 0, 'pos': 1}) if 'class' in test_df.columns else None

print(f"✅ Data loaded!")
print(f"Training features: {X_train_raw.shape}")
print(f"Training target: {y_train.shape}")
print(f"Test features: {X_test_raw.shape}")
if y_test is not None:
    print(f"Test target: {y_test.shape}")

📂 Loading data...
✅ Data loaded!
Training features: (60000, 170)
Training target: (60000,)
Test features: (16000, 170)
Test target: (16000,)


In [4]:
class MissingValueHandler:
    """Advanced missing value handler with multiple strategies"""
    
    def __init__(self, high_threshold=0.8):
        self.high_threshold = high_threshold
        self.feature_groups = {}
        self.imputers = {}
        self.binary_flags = {}
    
    def fit(self, X):
        """Fit missing value handler to data"""
        missing_pct = X.isna().sum() / len(X) * 100
        
        # Categorize features
        self.feature_groups['high'] = missing_pct[missing_pct >= self.high_threshold * 100].index.tolist()
        self.feature_groups['medium'] = missing_pct[(missing_pct >= 20) & (missing_pct < self.high_threshold * 100)].index.tolist()
        self.feature_groups['low'] = missing_pct[(missing_pct > 0) & (missing_pct < 20)].index.tolist()
        self.feature_groups['none'] = missing_pct[missing_pct == 0].index.tolist()
        
        # Create binary flags for high missing features
        for feature in self.feature_groups['high']:
            self.binary_flags[feature] = f"{feature}_missing_flag"
        
        # Fit medium missing imputer (MICE)
        if self.feature_groups['medium']:
            self.imputers['medium'] = IterativeImputer(
                max_iter=10,
                random_state=42,
                n_nearest_features=5
            )
            self.imputers['medium'].fit(X[self.feature_groups['medium']])
        
        # Fit low missing imputer (median)
        if self.feature_groups['low']:
            self.imputers['low'] = SimpleImputer(strategy='median')
            self.imputers['low'].fit(X[self.feature_groups['low']])
        
        return self
    
    def transform(self, X):
        """Transform data with missing value handling"""
        X_transformed = X.copy()
        
        # 1. Handle high missing: Create binary flags
        for feature in self.feature_groups['high']:
            flag_name = self.binary_flags[feature]
            X_transformed[flag_name] = X[feature].isna().astype(int)
            X_transformed = X_transformed.drop(columns=[feature])
        
        # 2. Handle medium missing: MICE imputation
        if self.feature_groups['medium']:
            X_transformed[self.feature_groups['medium']] = self.imputers['medium'].transform(
                X_transformed[self.feature_groups['medium']]
            )
        
        # 3. Handle low missing: Median imputation
        if self.feature_groups['low']:
            X_transformed[self.feature_groups['low']] = self.imputers['low'].transform(
                X_transformed[self.feature_groups['low']]
            )
        
        return X_transformed
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

print("✅ MissingValueHandler class defined!")

✅ MissingValueHandler class defined!


In [5]:
# Apply missing value handler
missing_handler = MissingValueHandler(high_threshold=0.8)
X_train_imputed = missing_handler.fit_transform(X_train_raw)
X_test_imputed = missing_handler.transform(X_test_raw)

print("✅ Missing value handling complete!")
print(f"Training shape: {X_train_imputed.shape}")
print(f"Test shape: {X_test_imputed.shape}")
print(f"Added {X_train_imputed.shape[1] - X_train_raw.shape[1]} missing flag features")

✅ Missing value handling complete!
Training shape: (60000, 170)
Test shape: (16000, 170)
Added 0 missing flag features


In [7]:
class HistogramFeatureEngineer:
    """Extract statistical features from histogram data"""
    
    def __init__(self, prefix='ay_'):
        self.prefix = prefix
        self.hist_features = []
    
    def fit(self, X):
        self.hist_features = [col for col in X.columns if col.startswith(self.prefix)]
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        if not self.hist_features:
            return X_transformed
        
        hist_data = X[self.hist_features].values
        
        # Create statistical features
        X_transformed[f'{self.prefix}mean'] = np.nanmean(hist_data, axis=1)
        X_transformed[f'{self.prefix}variance'] = np.nanvar(hist_data, axis=1)
        X_transformed[f'{self.prefix}std'] = np.nanstd(hist_data, axis=1)
        X_transformed[f'{self.prefix}skew'] = pd.DataFrame(hist_data).skew(axis=1)
        X_transformed[f'{self.prefix}kurtosis'] = pd.DataFrame(hist_data).kurtosis(axis=1)
        
        # Peak location
        peak_indices = np.nanargmax(hist_data, axis=1)
        X_transformed[f'{self.prefix}peak_index'] = peak_indices
        
        # Entropy
        hist_probs = hist_data / (np.nansum(hist_data, axis=1, keepdims=True) + 1e-10)
        entropy = -np.nansum(hist_probs * np.log(hist_probs + 1e-10), axis=1)
        X_transformed[f'{self.prefix}entropy'] = entropy
        
        # IQR
        hist_quantiles = np.percentile(hist_data, [25, 75], axis=1)
        X_transformed[f'{self.prefix}iqr'] = hist_quantiles[1] - hist_quantiles[0]
        
        # Peak-to-mean ratio
        peak_values = np.nanmax(hist_data, axis=1)
        mean_values = np.nanmean(hist_data, axis=1)
        X_transformed[f'{self.prefix}peak_ratio'] = peak_values / (mean_values + 1e-10)
        
        # Drop original histogram features
        X_transformed = X_transformed.drop(columns=self.hist_features)
        
        return X_transformed
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

# Apply histogram feature engineering
hist_engineer = HistogramFeatureEngineer(prefix='ay_')
X_train_hist = hist_engineer.fit_transform(X_train_imputed)
X_test_hist = hist_engineer.transform(X_test_imputed)

print(f"✅ Histogram features engineered!")
print(f"Added {X_train_hist.shape[1] - X_train_imputed.shape[1]} histogram-derived features")
print(f"New shape: {X_train_hist.shape}")

✅ Histogram features engineered!
Added -1 histogram-derived features
New shape: (60000, 169)


In [8]:
class CounterFeatureEngineer:
    """Engineer features from counter data"""
    
    def __init__(self, counter_groups=None):
        if counter_groups is None:
            self.counter_groups = ['ag_', 'az_', 'ba_', 'cs_', 'ee_', 'cn_']
        else:
            self.counter_groups = counter_groups
        self.groups = {}
    
    def fit(self, X):
        for prefix in self.counter_groups:
            self.groups[prefix] = [col for col in X.columns if col.startswith(prefix)]
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        for prefix, features in self.groups.items():
            if len(features) > 1:
                # Cumulative sum
                X_transformed[f'{prefix}cumsum'] = X[features].sum(axis=1)
                
                # Rate of change
                for i in range(len(features) - 1):
                    diff = X[features[i+1]] - X[features[i]]
                    X_transformed[f'{prefix}diff_{i}'] = diff.clip(lower=0)
                
                # Max/min ratio
                max_vals = X[features].max(axis=1)
                min_vals = X[features].min(axis=1)
                X_transformed[f'{prefix}max_min_ratio'] = max_vals / (min_vals + 1e-10)
                
                # Mean and std
                X_transformed[f'{prefix}mean'] = X[features].mean(axis=1)
                X_transformed[f'{prefix}std'] = X[features].std(axis=1)
                
                # Drop original counter features
                X_transformed = X_transformed.drop(columns=features)
        
        return X_transformed
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

# Apply counter feature engineering
counter_engineer = CounterFeatureEngineer()
X_train_counter = counter_engineer.fit_transform(X_train_hist)
X_test_counter = counter_engineer.transform(X_test_hist)

print(f"✅ Counter features engineered!")
print(f"Added {X_train_counter.shape[1] - X_train_hist.shape[1]} counter-derived features")
print(f"New shape: {X_train_counter.shape}")

✅ Counter features engineered!
Added 18 counter-derived features
New shape: (60000, 187)


In [11]:
class FeatureTransformer:
    """Apply transformations to features - OPTIMIZED VERSION"""
    
    def __init__(self, use_power_transform=False):
        """
        Args:
            use_power_transform: If True, apply PowerTransformer to skewed features.
                                 If False, only scale (much faster).
        """
        self.scaler = RobustScaler()
        self.transformer = None
        self.features_to_transform = []
        self.use_power_transform = use_power_transform
    
    def fit(self, X):
        # Identify skewed features (but only if we're using power transform)
        if self.use_power_transform:
            skewness = X.skew()
            self.features_to_transform = skewness[(skewness > 1) | (skewness < -1)].index.tolist()
            
            # Apply PowerTransformer to skewed features
            if self.features_to_transform:
                self.transformer = PowerTransformer(method='yeo-johnson')
                self.transformer.fit(X[self.features_to_transform])
                print(f"  Power transforming {len(self.features_to_transform)} skewed features")
        else:
            print("  Skipping power transform (using scaling only)")
        
        # Fit scaler
        self.scaler.fit(X)
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        # Transform skewed features (if enabled)
        if self.transformer and self.features_to_transform:
            X_transformed[self.features_to_transform] = self.transformer.transform(
                X_transformed[self.features_to_transform]
            )
        
        # Scale all features
        X_scaled = self.scaler.transform(X_transformed)
        X_transformed = pd.DataFrame(
            X_scaled,
            columns=X_transformed.columns,
            index=X_transformed.index
        )
        
        return X_transformed
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)


# Apply feature transformation - WITH FAST MODE
print("🔄 Applying feature transformation (fast mode)...")

# Option 1: FAST - No power transform (just scaling)
transformer = FeatureTransformer(use_power_transform=False)
X_train_transformed = transformer.fit_transform(X_train_counter)
X_test_transformed = transformer.transform(X_test_counter)

print(f"\n✅ Feature transformation complete!")
print(f"  Transformed {len(transformer.features_to_transform)} skewed features")
print(f"  Final shape: {X_train_transformed.shape}")

# Option 2: If you want power transform, uncomment below (slower but better)
"""
# Option 2: SLOWER but better for skewed features
transformer = FeatureTransformer(use_power_transform=True)
X_train_transformed = transformer.fit_transform(X_train_counter)
X_test_transformed = transformer.transform(X_test_counter)
"""

🔄 Applying feature transformation (fast mode)...
  Skipping power transform (using scaling only)

✅ Feature transformation complete!
  Transformed 0 skewed features
  Final shape: (60000, 187)


'\n# Option 2: SLOWER but better for skewed features\ntransformer = FeatureTransformer(use_power_transform=True)\nX_train_transformed = transformer.fit_transform(X_train_counter)\nX_test_transformed = transformer.transform(X_test_counter)\n'

In [13]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier

class FastFeatureSelector:
    """Fast feature selection with multiple strategies"""
    
    def __init__(self, variance_threshold=0.01, k_features=100, method='f_classif'):
        """
        Args:
            variance_threshold: Threshold for variance filtering
            k_features: Number of features to select
            method: 'f_classif' (fast), 'mutual_info' (slower but better), 'random_forest' (medium)
        """
        self.variance_threshold = variance_threshold
        self.k_features = k_features
        self.method = method
        self.selected_features = []
        self.selector = None
        self.scores = None
    
    def fit(self, X, y):
        print(f"  Using method: {self.method}")
        
        # 1. Remove low variance features (fast)
        print("  Removing low variance features...")
        variance_filter = VarianceThreshold(threshold=self.variance_threshold)
        variance_filter.fit(X)
        high_variance_features = X.columns[variance_filter.get_support()].tolist()
        print(f"    Kept {len(high_variance_features)} features")
        
        # 2. Select top k features
        k = min(self.k_features, len(high_variance_features))
        X_high_var = X[high_variance_features]
        
        if self.method == 'f_classif':
            # FASTEST: ANOVA F-test
            print("  Using F-test for feature selection...")
            self.selector = SelectKBest(f_classif, k=k)
            self.selector.fit(X_high_var, y)
            self.scores = self.selector.scores_
            
        elif self.method == 'random_forest':
            # MEDIUM: Random Forest feature importance
            print("  Using Random Forest for feature selection...")
            rf = RandomForestClassifier(
                n_estimators=50,  # Lower for speed
                max_depth=5,
                random_state=42,
                n_jobs=-1
            )
            rf.fit(X_high_var, y)
            importances = rf.feature_importances_
            
            # Get top k features
            top_indices = np.argsort(importances)[-k:][::-1]
            self.selected_features = [high_variance_features[i] for i in top_indices]
            self.scores = importances[top_indices]
            return self
            
        elif self.method == 'mutual_info':
            # SLOWER but better for non-linear relationships
            print("  Using Mutual Information (this may take a while)...")
            from sklearn.feature_selection import mutual_info_classif
            mi_scores = mutual_info_classif(X_high_var, y, random_state=42)
            top_indices = np.argsort(mi_scores)[-k:][::-1]
            self.selected_features = [high_variance_features[i] for i in top_indices]
            self.scores = mi_scores[top_indices]
            return self
        
        # Get selected features for f_classif
        selected_indices = self.selector.get_support(indices=True)
        self.selected_features = [high_variance_features[i] for i in selected_indices]
        self.scores = self.selector.scores_[selected_indices]
        
        return self
    
    def transform(self, X):
        return X[self.selected_features]
    
    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)


# Apply fast feature selection
print("🔄 Applying feature selection (fast mode)...")
feature_selector = FastFeatureSelector(
    variance_threshold=0.01, 
    k_features=100,
    method='f_classif'  # FASTEST option
)

X_train_selected = feature_selector.fit_transform(X_train_transformed, y_train)
X_test_selected = feature_selector.transform(X_test_transformed)

print(f"\n✅ Feature selection complete!")
print(f"Selected {len(feature_selector.selected_features)} features")
print(f"Final shape: {X_train_selected.shape}")

# Show top selected features
selected_feature_info = pd.DataFrame({
    'Feature': feature_selector.selected_features,
    'Score': feature_selector.scores
}).sort_values('Score', ascending=False)

print("\n📊 TOP 20 SELECTED FEATURES:")
for idx, row in selected_feature_info.head(20).iterrows():
    print(f"  {row['Feature']}: {row['Score']:.4f}")

🔄 Applying feature selection (fast mode)...
  Using method: f_classif
  Removing low variance features...
    Kept 185 features
  Using F-test for feature selection...

✅ Feature selection complete!
Selected 100 features
Final shape: (60000, 100)

📊 TOP 20 SELECTED FEATURES:
  ci_000: 26073.0237
  aa_000: 24309.7993
  bt_000: 23969.5107
  bb_000: 23527.8125
  bv_000: 23354.4171
  bu_000: 23354.4142
  cq_000: 23354.4121
  aq_000: 22189.5061
  bj_000: 21531.5116
  ah_000: 21428.0086
  cc_000: 21408.6188
  an_000: 21333.7259
  bg_000: 21172.1090
  ao_000: 20938.8147
  bx_000: 20596.7537
  ap_000: 20392.1496
  by_000: 20014.8128
  bh_000: 18525.4722
  dn_000: 18151.2712
  ba_mean: 17456.7315


In [14]:
# Create directories if they don't exist
import os
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save processed features
joblib.dump(X_train_selected, '../data/processed/X_train.pkl')
joblib.dump(X_test_selected, '../data/processed/X_test.pkl')
joblib.dump(y_train, '../data/processed/y_train.pkl')
if y_test is not None:
    joblib.dump(y_test, '../data/processed/y_test.pkl')

# Save the complete preprocessing pipeline for deployment
preprocessing_pipeline = {
    'missing_handler': missing_handler,
    'hist_engineer': hist_engineer,
    'counter_engineer': counter_engineer,
    'transformer': transformer,
    'selector': feature_selector
}
joblib.dump(preprocessing_pipeline, '../models/preprocessing_pipeline.pkl')

print("✅ All processed data saved!")
print(f"  - X_train: {X_train_selected.shape}")
print(f"  - X_test: {X_test_selected.shape}")
print(f"  - y_train: {y_train.shape}")
if y_test is not None:
    print(f"  - y_test: {y_test.shape}")
print(f"  - Pipeline: saved to models/preprocessing_pipeline.pkl")

✅ All processed data saved!
  - X_train: (60000, 100)
  - X_test: (16000, 100)
  - y_train: (60000,)
  - y_test: (16000,)
  - Pipeline: saved to models/preprocessing_pipeline.pkl


In [15]:
print("="*80)
print("📊 FEATURE ENGINEERING SUMMARY")
print("="*80)

print(f"\n📁 Original vs Processed:")
print(f"  Original features: {X_train_raw.shape[1]}")
print(f"  Processed features: {X_train_selected.shape[1]}")
print(f"  Reduction: {X_train_raw.shape[1] - X_train_selected.shape[1]} features removed")

print(f"\n🔍 Missing Values (After Processing):")
print(f"  Total missing: {X_train_selected.isna().sum().sum()}")
print(f"  Missing percentage: {X_train_selected.isna().sum().sum() / (X_train_selected.shape[0] * X_train_selected.shape[1]) * 100:.4f}%")

print(f"\n📊 Feature Diversity:")
print(f"  Original features: {X_train_raw.shape[1]}")
print(f"  Engineered features added: {X_train_selected.shape[1] - X_train_raw.shape[1]}")
print(f"  Final feature set: {X_train_selected.shape[1]}")

print("\n✅ Phase 2 Complete! Ready for Model Development!")

📊 FEATURE ENGINEERING SUMMARY

📁 Original vs Processed:
  Original features: 170
  Processed features: 100
  Reduction: 70 features removed

🔍 Missing Values (After Processing):
  Total missing: 0
  Missing percentage: 0.0000%

📊 Feature Diversity:
  Original features: 170
  Engineered features added: -70
  Final feature set: 100

✅ Phase 2 Complete! Ready for Model Development!
